# Parlay SFT (Qwen2.5-1.5B) — Colab

Runs [`training/sft_train.py`](https://github.com/sh4shv4t/Parlay/blob/main/training/sft_train.py) from your repo (clone `sh4shv4t/Parlay`).

**Run order:** config → (Drive) → **HF token** (if you push, or a *private* Hub dataset) → **Environment** (pip + optional `JSONL_VIA_HF` download) → clone → train → push.

## Data (`data/` is gitignored on GitHub)
Episode `*.jsonl` is **not** in the GitHub repo. Use one of:

1. **Google Drive:** e.g. `MyDrive/ParlaySFT/episodes_v2.jsonl` → set `JSONL_IN_DRIVE` in the config cell and mount Drive.
2. **Hugging Face Dataset repo:** e.g. `sh4shv4t/parlay-episodes` (create at [huggingface.co/new-dataset](https://huggingface.co/new-dataset)), upload `episodes_v2.jsonl` on the **Files** tab, set `JSONL_VIA_HF = ("sh4shv4t/parlay-episodes", "episodes_v2.jsonl")`, then run the **Environment** cell.
3. **Colab Files:** upload `episodes_v2.jsonl` to `/content/` and set `EPISODES_JSONL = "/content/episodes_v2.jsonl"`.

## Model output
Use a **Hugging Face Model** repo for the LoRA (e.g. `sh4shv4t/parlay-sft-1-5b`). This is separate from a Git code repo named `Parlay` so your Hub stays organized.

## Colab + CUDA
Use **Runtime → Change runtime type → GPU.** Colab’s `torch` is already built for its CUDA driver; this notebook only `pip install`s Transformers, TRL, and related libs (do not `pip install torch` unless you know you need a different build).

In [ ]:
# --- Set these, then run all cells in order (top to bottom) ---
GITHUB_CLONE = "https://github.com/sh4shv4t/Parlay.git"
REPO_DIR = "/content/Parlay"

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MIN_REWARD = -50.0
# If you do not use Drive, use e.g. "/content/sft_1.5b"
OUTPUT_DIR = "/content/drive/MyDrive/ParlaySFT/sft_1.5b"

EPISODES_JSONL = "/content/episodes_v2.jsonl"
JSONL_IN_DRIVE = None  # e.g. "/content/drive/MyDrive/ParlaySFT/episodes_v2.jsonl"
JSONL_VIA_HF = None  # e.g. ("sh4shv4t/parlay-episodes", "episodes_v2.jsonl")

HF_MODEL_REPO = "sh4shv4t/parlay-sft-1-5b"
PUSH_TO_HF = True

from pathlib import Path as _P
if JSONL_IN_DRIVE:
    EPISODES_JSONL = str(_P(JSONL_IN_DRIVE))
print("Config OK.")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab; skip Drive or mount manually.")

In [ ]:
import getpass
import os
import subprocess

def _get_hf_token() -> str:
    for k in ("HUGGINGFACE_HUB_TOKEN", "HF_TOKEN"):
        v = os.environ.get(k) or ""
        if v:
            return v
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    if PUSH_TO_HF or JSONL_VIA_HF:
        return getpass.getpass("HuggingFace token (needed for private hub assets / push): ")
    return ""

if PUSH_TO_HF or JSONL_VIA_HF:
    _t = _get_hf_token()
    if _t:
        os.environ["HF_TOKEN"] = _t
        os.environ["HUGGINGFACE_HUB_TOKEN"] = _t
        print("HF token set (len %d)" % len(_t))
    else:
        print("Warning: no HF token; private dataset download or push may fail.")
else:
    print("No HF token required (set PUSH/JSONL_VIA_HF to require).")

In [ ]:
import importlib.util
import subprocess
import sys
def _in_colab() -> bool:
    return importlib.util.find_spec("google.colab") is not None
print("Colab:", _in_colab())
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" |", torch.cuda.get_device_name(0))
else:
    print()

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.47.0", "trl>=0.14.0", "peft>=0.14.0", "accelerate>=1.2.0",
    "datasets>=3.2.0", "huggingface-hub>=0.24.0",
])

if JSONL_VIA_HF:
    from huggingface_hub import hf_hub_download
    _repo, _fn = JSONL_VIA_HF
    EPISODES_JSONL = hf_hub_download(repo_id=_repo, filename=_fn, repo_type="dataset")
    print("Dataset file:", EPISODES_JSONL)
else:
    print("EPISODES_JSONL:", EPISODES_JSONL)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

rd = Path(REPO_DIR)
sft = rd / "training" / "sft_train.py"
if not sft.is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_CLONE, str(rd)])
os.chdir(rd)
sys.path.insert(0, str(rd))
print("CWD:", os.getcwd(), "| sft_train:", sft.is_file())

In [ ]:
import json
import os
import time
from pathlib import Path
from training.sft_train import train_sft, load_sft_dataset

data_path = Path(EPISODES_JSONL).resolve()
if not data_path.is_file():
    raise FileNotFoundError(
        f"No JSONL at {data_path}. Set Drive/HF/Colab upload paths in the config cell."
    )
out = Path(OUTPUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)
ds = load_sft_dataset(data_path, min_reward=MIN_REWARD)
n = len(ds)
print(f"SFT text rows: {n}")
t0 = time.time()
train_sft(data_path, BASE_MODEL, out, min_reward=MIN_REWARD)
elapsed = time.time() - t0
summary = {
  "base_model": BASE_MODEL,
  "episodes_jsonl": str(data_path),
  "output_dir": str(out),
  "n_sft_text_rows": n,
  "min_reward": MIN_REWARD,
  "train_seconds": round(elapsed, 1),
  "hf_model_repo": HF_MODEL_REPO if PUSH_TO_HF else None,
  "model_page_url": f"https://huggingface.co/{HF_MODEL_REPO}" if PUSH_TO_HF else None,
}
sp = out / "sft_colab_run_summary.json"
sp.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(sp.read_text())

In [ ]:
import os
from pathlib import Path
if not PUSH_TO_HF:
    print("PUSH_TO_HF is False; results are on disk in OUTPUT_DIR only.")
else:
    from huggingface_hub import HfApi
    tok = os.environ.get("HUGGINGFACE_HUB_TOKEN") or os.environ.get("HF_TOKEN")
    if not tok:
        raise ValueError("Set HF token (Colab secret HF_TOKEN) before push.")
    out = str(Path(OUTPUT_DIR).resolve())
    api = HfApi(token=tok)
    api.create_repo(repo_id=HF_MODEL_REPO, private=False, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path=out,
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        commit_message="Parlay SFT LoRA (Qwen2.5-1.5B) from Colab",
    )
    print("Model page:", f"https://huggingface.co/{HF_MODEL_REPO}")

## Inference on Hub / “endpoint”
- The **URL** for your weights is `https://huggingface.co/<username>/<model-repo>` (e.g. `sh4shv4t/parlay-sft-1-5b`) after push.
- For **Serverless Inference API**, open the model on Hub → *Settings* / *Deploy* (availability depends on account and model size). For a dedicated endpoint, use **Inference Endpoints** from the same org.

Load the adapter in Python (base + LoRA):
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
m = AutoModelForCausalLM.from_pretrained(base, torch_dtype="auto", device_map="auto")
m = PeftModel.from_pretrained(m, "sh4shv4t/parlay-sft-1-5b")
```
If you see **CUDA OOM**, in `training/sft_train.py` reduce `per_device_train_batch_size` and increase `gradient_accumulation_steps`.